# Understanding Learning Curves

Now that you've learned how to implement fine-tuning using both the `Trainer` API and custom training loops, it's crucial to understand how to interpret the results. Learning curves are invaluable tools that help you evaluate your model's performance during training and identify potential issues before they reduce performance.


Let's explore how to read and interpret accuracy and loss curves, understand what different curve shapes tell us about our model's behavior, and learn how to address common traning issues.

## What are Learning Curves?

Learning curves are visual representations of your model's performance metrics over time during training. The two most important curves to monitor are:
- **Loss curves**: Show how the model's error (loss) changes over training steps or epochs
- **Accuracy curves**: Shows the percentage of correct predictions over training steps or epochs.

These curves help us understand whether our model is learning effectively and can guide us in making adjustments to improve performance. In Transformers, these metrics are individually computed for each batch and then logged to the disk. We can then use libraries like *Weights & Biases** to visualize these curves and track our model's performance over time.

### Loss Curves

The loss curve shows how the model's error decreases over time.

In a typical successful training run, you'll see a curve similar to the one below:

![](./resources/loss_curve.png)
*Image source: [Hugging Face LLM Course, Chapter 3.5](https://huggingface.co/learn/llm-course/chapter3/5?fw=pt)*

- **High initial loss**: The model starts without optimization, so predictions are initially poor.
- **Decreasing loss:** As training progresses, the loss should generally decrease
- **Convergence:** Eventually, the loss stabilizes at a low value, indicating that the model has learned the patterns in the data.

We can use the `Trainer` API to track these metrics and visualize them in a dashboard. Below is an example of how to do this with **Weights & Biases**.

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

import evaluate

metric=evaluate.load('glue','mrpc')

import numpy as np
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Example of tracking loss during training with the Trainer
from transformers import Trainer, TrainingArguments
import wandb
# Initialize Weights & Biases for experiment tracking
wandb.init(project="transformer-fine-tuning", name="bert-mrpc-analysis")

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    logging_steps=10,  # Log metrics every 10 steps
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    report_to="wandb",  # Send logs to Weights & Biases
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train and automatically log metrics
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1
50,0.582700,0.598143,0.671569,0.726531
100,0.515500,0.481190,0.767157,0.851794
150,0.492800,0.444436,0.796569,0.868045
200,0.381500,0.396500,0.843137,0.891892
250,0.259600,0.362040,0.840686,0.881603
300,0.231800,0.407079,0.865196,0.904679
350,0.234100,0.340227,0.865196,0.903339
400,0.133400,0.422422,0.850490,0.897133
450,0.269100,0.340626,0.870098,0.905526
500,0.044000,0.524982,0.857843,0.902685


/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/dimitaratanassov/huggingface_workspace/hf_llmcourse_venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argum

TrainOutput(global_step=690, training_loss=0.28975238411322884, metrics={'train_runtime': 181.9147, 'train_samples_per_second': 60.49, 'train_steps_per_second': 3.793, 'total_flos': 428577075854640.0, 'train_loss': 0.28975238411322884, 'epoch': 3.0})

### Accuracy Curves

The accuracy curve shows the percentage of correct predictions over time. Unlike loss curves, accuracy curves should generally increase as the model learns and can typically include more steps than loss curve.

![](./resources/accuracy_curve.png)

*Image source: [Hugging Face LLM Course, Chapter 3.5](https://huggingface.co/learn/llm-course/chapter3/5?fw=pt)*

- **Start Low**: Initial accuracy should be low, as the model has not yet learned the patterns in the data
- **Increase with training**: Accuracy should generally improve as the model learns if it is able to learn the patterns in the data.
- **May show plateaus**: Accuracy often increases in discrete jumps rather than smoothly, as the model makes predictions that are close to the true labels